In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister, IfElseOp 
from qiskit.circuit.classical import expr
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)

circuit = QuantumCircuit(qubits, clbits) 
(q0, q1) = qubits
(c0, c1) = clbits
circuit.barrier() 
circuit.h(q0) 
circuit.measure(q0, c0) 
with circuit.if_test((c0, 1)):
    circuit.x(q0)

# Apply an XX DD sequence with stretch on qubit 1
a = circuit.add_stretch("a") 
circuit.delay(a, q1) 
circuit.x(q1) 
circuit.delay(expr.div(a, 2), q1) 
circuit.x(q1)
circuit.delay(a, q1) 
circuit.barrier()

# Patch the backend target with IfElseOp
service = QiskitRuntimeService()
backend = service.backend("ibm_fez") 
backend.target.add_instruction(IfElseOp, name="if_else")

# The target can now be used for transpilation
pm = generate_preset_pass_manager(optimization_level=1, target=backend.target) 
isa_circuit = pm.run(circuit)

# Submit the job with the experimental option
sampler = SamplerV2(backend)
sampler.options.experimental = {"execution_path" : "gen3-experimental"} 
job = sampler.run([isa_circuit])
result = job.result()

In [2]:
print(result)

PrimitiveResult([SamplerPubResult(data=DataBin(c0=BitArray(<shape=(), num_shots=4096, num_bits=2>)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2025-07-29 17:51:42', stop='2025-07-29 17:51:48', size=4096>)])}, 'version': 2})


In [3]:
print(isa_circuit)

global phase: π/4
           ░ »
q0_0 -> 0 ─░─»
           ░ »
q0_1 -> 1 ─░─»
           ░ »
    c0: 2/═══»
             »
«                                          ┌─────────┐                                »
«q0_0 -> 0 ────────────────────────────────┤ Rz(π/2) ├────────────────────────────────»
«          ┌───────────────────────────────┴─────────┴───────────────────────────────┐»
«q0_1 -> 1 ┤ Delay(Stretch(UUID('008fb19f-6ff3-413f-9b94-ad54741e9cd4'), 'a')[expr]) ├»
«          └─────────────────────────────────────────────────────────────────────────┘»
«    c0: 2/═══════════════════════════════════════════════════════════════════════════»
«                                                                                     »
«          ┌────┐»
«q0_0 -> 0 ┤ √X ├»
«          ├───┬┘»
«q0_1 -> 1 ┤ X ├─»
«          └───┘ »
«    c0: 2/══════»
«                »
«                                                                        ┌─────────┐                                            

In [10]:
print(pub_result = job_result[0].data.clbits.get_counts())

AttributeError: 'PrimitiveResult' object has no attribute 'get_counts'